# Three-dimensional tetrahedral MHM

An exact affine patch followed by the archived five-level sinusoidal cube study. P2 local pressure and four independent P0 subtriangles per macroface. The raw gradient flux is broken; skeleton flux is macro conservative.


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.tetrahedral import TetraMesh
from pymhm.darcy3d import TriangularSkeleton, solve_darcy_3d
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent


In [ ]:
def pressure(points):
    """Return a nonhomogeneous affine pressure patch."""
    return 1 + points @ np.array([1., 2., 3.])

mesh = TetraMesh.unit_cube(1)
solution = solve_darcy_3d(mesh, skeleton=TriangularSkeleton(mesh, 2), degree=2,
                          local_refinement=2, dirichlet=pressure)
assert solution.l2_error(pressure) < 1e-12
assert solution.flux_l2_error([-1., -2., -3.]) < 1e-11
assert np.max(np.abs(solution.conservation_residuals())) < 1e-12
(solution.l2_error(pressure), solution.flux_l2_error([-1., -2., -3.]))


## Recorded convergence

The PDE matches the sinusoidal cube example in Gomes et al. (2017), section 5. The Freudenthal macro meshes and red-refined local meshes differ from its TetGen meshes; these data verify convergence and do not reproduce cluster timing tables. Volume norms use the full P2 function and gradient.


In [ ]:
report = json.loads((root / "examples/results/darcy3d.json").read_text())
assert len(report["rows"]) == 5
assert not report["source_changed_during_run"]
for row in report["rows"]:
    assert row["macro_balance_max"] < 1e-12
[(r["macro_tetrahedra"], r["pressure_l2_error"], r["flux_l2_error"]) for r in report["rows"]]


In [ ]:
display(Image(filename=str(root / "docs/figures/darcy3d/convergence.png")))
display(Image(filename=str(root / "docs/figures/darcy3d/fields.png")))


The cross-section preserves independent cut-cell values and highlights the actual macro intersections. Flat shading samples each cut-cell centroid; it does not replace the volumetric norm integration. Recompute with `examples/solve_darcy3d.py` and render with `examples/plot_darcy3d.py`.
